# Chapter 03: Workflow YAML Anatomy and Your First Green Check (notebook edition)

## Learning Objectives

- Parse a workflow into triggers and steps
- Predict whether a push fires it
- Read green vs red step conclusions

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. Parse the real workflow

We load `ch03-tally-ci.yml` with our `on`-aware loader and list its triggers. You should see `push` and `workflow_dispatch`.

In [ ]:
from intro_gha import WORKFLOWS_DIR
from intro_gha.workflow import load_workflow, triggers
wf = load_workflow(WORKFLOWS_DIR / "ch03-tally-ci.yml")
trig = triggers(wf)
print(sorted(trig))
assert set(trig) == {"push", "workflow_dispatch"}

## 2. Test the paths filter

`paths` fires when ANY changed file matches. We try four change sets.

In [ ]:
from intro_gha.triggers import paths_trigger
paths = trig["push"]["paths"]
cases = {("sandbox/tally/tally/add.py",): True, ("README.md",): False,
         ("README.md","sandbox/tally/tests/test_add.py"): True,
         ("learning_modules/chapter_03_workflow_anatomy.md",): False}
for changed, expected in cases.items():
    print(changed, paths_trigger(list(changed), paths))
    assert paths_trigger(list(changed), paths) is expected

## 3. Read the green and red runs

The fixtures hold each step's conclusion. The `Break add()` step is `skipped` on green and `success` on red; `Run tests` flips.

In [ ]:
import json
def steps(name):
    run = json.loads((ROOT / "fixtures" / f"run_ch03_{name}.json").read_text())
    return {s["name"]: s["conclusion"] for s in run["steps"]}
green, red = steps("green"), steps("red")
print(green["Break add() on purpose"], red["Break add() on purpose"], green["Run tests"], red["Run tests"])
assert (green["Break add() on purpose"], red["Break add() on purpose"]) == ("skipped", "success")
assert (green["Run tests"], red["Run tests"]) == ("success", "failure")

## Takeaways & Next Steps

- `if:` false is skipped, not failed.
- `paths` fires on any match; no match means no run.
- Next: Chapter 04, what a `uses:` reference really is.

## Chapter Link

Read: `learning_modules/chapter_03_workflow_anatomy.md`